In [1]:
import os
from pathlib import Path
import pandas as pd

DATASET_DIR = Path("/kaggle/input/finetunewavlm-v1")

# manifest
MANIFEST = DATASET_DIR / "manifest_strict.csv"

# audio folder (punyamu double preprocessed_full)
AUDIO_DIR = DATASET_DIR / "preprocessed_full" / "preprocessed_full"

print("MANIFEST:", MANIFEST)
print("AUDIO_DIR:", AUDIO_DIR)
print("exists?", MANIFEST.exists(), AUDIO_DIR.exists())

df_all = pd.read_csv(MANIFEST)
df_all["clip_id"] = df_all["clip_id"].astype(str)

# split strict
spl = df_all["split_strict"].astype(str).str.lower()
df_train = df_all[spl == "train"].reset_index(drop=True)
df_val   = df_all[spl == "val"].reset_index(drop=True)
df_test  = df_all[spl == "test"].reset_index(drop=True)  # locked

label_cols = ["extraversion","neuroticism","agreeableness","conscientiousness","openness"]

# build audio_path
df_train["audio_path"] = df_train["clip_id"].map(lambda x: str(AUDIO_DIR / f"{x}.wav"))
df_val["audio_path"]   = df_val["clip_id"].map(lambda x: str(AUDIO_DIR / f"{x}.wav"))
df_test["audio_path"]  = df_test["clip_id"].map(lambda x: str(AUDIO_DIR / f"{x}.wav"))

print("shapes:", df_train.shape, df_val.shape, df_test.shape)

# quick check audio exists
missing = df_train[~df_train["audio_path"].map(lambda p: Path(p).exists())]
print("missing train wav:", len(missing))
print(df_train[["clip_id","audio_path"]].head())


MANIFEST: /kaggle/input/finetunewavlm-v1/manifest_strict.csv
AUDIO_DIR: /kaggle/input/finetunewavlm-v1/preprocessed_full/preprocessed_full
exists? True True
shapes: (5936, 15) (1999, 15) (2039, 15)
missing train wav: 0
           clip_id                                         audio_path
0  --Ymqszjv54.000  /kaggle/input/finetunewavlm-v1/preprocessed_fu...
1  --Ymqszjv54.001  /kaggle/input/finetunewavlm-v1/preprocessed_fu...
2  --Ymqszjv54.003  /kaggle/input/finetunewavlm-v1/preprocessed_fu...
3  --Ymqszjv54.004  /kaggle/input/finetunewavlm-v1/preprocessed_fu...
4  --Ymqszjv54.005  /kaggle/input/finetunewavlm-v1/preprocessed_fu...


In [5]:
import os, random, math, time
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn

# (opsional) kurangi fragmentasi memori CUDA
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

HF_CACHE = Path("/kaggle/working/hf_cache")
HF_CACHE.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(HF_CACHE)
os.environ["HF_HUB_CACHE"] = str(HF_CACHE / "hub")
os.environ["TRANSFORMERS_CACHE"] = str(HF_CACHE / "transformers")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

from transformers import AutoFeatureExtractor, WavLMModel
from peft import LoraConfig, TaskType, get_peft_model

MODEL_NAME = "microsoft/wavlm-base-plus"

# LoRA
LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05
TARGET_MODULES = ["q_proj", "v_proj"]

LR = 1e-4
WEIGHT_DECAY = 0.01

feature_extractor = AutoFeatureExtractor.from_pretrained(MODEL_NAME, cache_dir=str(HF_CACHE))

# Load backbone float16 biar hemat VRAM
backbone = WavLMModel.from_pretrained(
    MODEL_NAME,
    cache_dir=str(HF_CACHE),
    torch_dtype=torch.float16,      # penting
    low_cpu_mem_usage=True
)

# freeze backbone
for p in backbone.parameters():
    p.requires_grad = False

# attach LoRA (ini tetap inject LoRA ke modul di dalam backbone)
lora_cfg = LoraConfig(
    task_type=TaskType.FEATURE_EXTRACTION,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=TARGET_MODULES,
    bias="none",
)
backbone = get_peft_model(backbone, lora_cfg)

# gradient checkpointing (aktifkan di core model setelah peft)
core = backbone.base_model.model if hasattr(backbone, "base_model") else backbone
core.gradient_checkpointing_enable()
core.config.use_cache = False

# ===== Model regressor + CHUNKED forward =====
TARGET_SR = getattr(feature_extractor, "sampling_rate", 16000)
MAX_SECONDS = 15
MAX_LEN = int(TARGET_SR * MAX_SECONDS)

CHUNK_SECONDS = 3                 # <- ini yang bikin jauh lebih hemat
CHUNK_LEN = int(TARGET_SR * CHUNK_SECONDS)

    def __init__(self, backbone_peft: nn.Module, num_labels: int = 5):
        super().__init__()
        self.backbone = backbone_peft
        self.core = self.backbone.base_model.model if hasattr(self.backbone, "base_model") else self.backbone
        h = self.core.config.hidden_size

        self.head = nn.Sequential(
            nn.Linear(h, h),
            nn.Tanh(),
            nn.Dropout(0.1),
            nn.Linear(h, num_labels),
        )
        self.loss_fn = nn.MSELoss()

    def _pool(self, x, attention_mask_wav):
        # x: (B, T_feat, H)
        if attention_mask_wav is None or not hasattr(self.core, "_get_feature_vector_attention_mask"):
            return x.mean(dim=1), torch.ones((x.size(0), 1), device=x.device, dtype=x.dtype)

        feat_mask = self.core._get_feature_vector_attention_mask(x.shape[1], attention_mask_wav)  # (B, T_feat)
        w = feat_mask.sum(dim=1).clamp(min=1).to(x.dtype).unsqueeze(-1)                            # (B,1)
        pooled = (x * feat_mask.unsqueeze(-1).to(x.dtype)).sum(dim=1) / w
        return pooled, w

    def forward(self, input_values, attention_mask=None, labels=None):
        # input_values: (B, T_wav)
        B, T = input_values.shape

        sum_vec = None
        sum_w = None

        # proses 15 detik, tapi per CHUNK_SECONDS
        for start in range(0, T, CHUNK_LEN):
            end = min(T, start + CHUNK_LEN)
            chunk = input_values[:, start:end]
            chunk_mask = attention_mask[:, start:end] if attention_mask is not None else None

            out = self.core(input_values=chunk, attention_mask=chunk_mask)
            x = out.last_hidden_state  # (B, T_feat, H)

            pooled, w = self._pool(x, chunk_mask)  # (B,H), (B,1)

            if sum_vec is None:
                sum_vec = pooled * w
                sum_w = w
            else:
                sum_vec = sum_vec + pooled * w
                sum_w = sum_w + w

        pooled_all = sum_vec / sum_w.clamp(min=1.0)
        preds = self.head(pooled_all)

        loss = self.loss_fn(preds, labels) if labels is not None else None
        return {"loss": loss, "preds": preds}

# init model
label_cols = ["extraversion","neuroticism","agreeableness","conscientiousness","openness"]
model = WavLMRegressorChunked(backbone, num_labels=len(label_cols)).to(DEVICE)

trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable_params, lr=LR, weight_decay=WEIGHT_DECAY)

tot = sum(p.numel() for p in model.parameters())
trn = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"params total={tot:,} | trainable={trn:,}")

RUN_NAME = f"wavlm_strict_chunk{CHUNK_SECONDS}s_lora_r{LORA_R}_lr{LR}_seed{SEED}"
OUT_ROOT = Path("/kaggle/working/output") / "finetune_strict" / RUN_NAME
CKPT_DIR = OUT_ROOT / "checkpoints"
CKPT_DIR.mkdir(parents=True, exist_ok=True)
print("OUT_ROOT:", OUT_ROOT)


DEVICE: cuda


/usr/local/lib/python3.12/dist-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
2026-01-18 12:04:05.434392: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768737845.785391      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768737845.890477      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1768737846.734442      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768737846.734482      55 computation_pl

preprocessor_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/378M [00:00<?, ?B/s]

params total=95,271,285 | trainable=889,349
OUT_ROOT: /kaggle/working/output/finetune_strict/wavlm_strict_chunk3s_lora_r8_lr0.0001_seed42


model.safetensors:   0%|          | 0.00/378M [00:00<?, ?B/s]

In [6]:
import soundfile as sf
from torch.utils.data import Dataset, DataLoader
import numpy as np
import torch

MAX_LEN = int(TARGET_SR * 15)

class StrictAudioDataset(Dataset):
    def __init__(self, df, label_cols):
        self.df = df.reset_index(drop=True)
        self.label_cols = label_cols

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        audio, sr = sf.read(row["audio_path"])
        if audio.ndim > 1:
            audio = audio.mean(axis=1)
        audio = audio.astype(np.float32)

        if sr != TARGET_SR:
            raise RuntimeError(f"SR mismatch: got {sr}, expected {TARGET_SR}")

        labels = row[self.label_cols].to_numpy(dtype=np.float32)
        return {"audio": audio, "labels": labels}

def collate_fn(batch):
    audios = [b["audio"] for b in batch]
    labels = torch.from_numpy(np.stack([b["labels"] for b in batch])).float()

    feats = feature_extractor(
        audios,
        sampling_rate=TARGET_SR,
        padding="longest",
        truncation=True,
        max_length=MAX_LEN,              # <- tetap 15 detik
        return_attention_mask=True,
        return_tensors="pt",
    )
    feats["labels"] = labels
    return feats

BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 16   # effective batch = 16
NUM_WORKERS = 0

train_loader = DataLoader(
    StrictAudioDataset(df_train, label_cols),
    batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, collate_fn=collate_fn,
    pin_memory=True
)
val_loader = DataLoader(
    StrictAudioDataset(df_val, label_cols),
    batch_size=1, shuffle=False,
    num_workers=NUM_WORKERS, collate_fn=collate_fn,
    pin_memory=True
)

print("loader ok | train steps:", len(train_loader), "| val steps:", len(val_loader))


loader ok | train steps: 5936 | val steps: 1999


In [ ]:

import gc, math, pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

use_amp = (DEVICE == "cuda")
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

MAX_EPOCHS = 20
PATIENCE = 5
GRAD_CLIP = 1.0

def to_device(batch, device):
    return {k: (v.to(device, non_blocking=True) if torch.is_tensor(v) else v) for k, v in batch.items()}

@torch.inference_mode()
def run_eval(model, loader):
    model.eval()
    all_preds, all_true = [], []
    total_loss, n_batches = 0.0, 0

    for batch in loader:
        batch = to_device(batch, DEVICE)
        labels = batch["labels"]
        with torch.cuda.amp.autocast(enabled=use_amp):
            out = model(
                input_values=batch["input_values"],
                attention_mask=batch.get("attention_mask", None),
                labels=labels
            )
            loss = out["loss"]

        total_loss += float(loss.item())
        n_batches += 1
        all_preds.append(out["preds"].detach().cpu().numpy())
        all_true.append(labels.detach().cpu().numpy())

    y_pred = np.concatenate(all_preds, axis=0)
    y_true = np.concatenate(all_true, axis=0)

    mae_list, rmse_list, r2_list = [], [], []
    for i in range(len(label_cols)):
        mae  = mean_absolute_error(y_true[:, i], y_pred[:, i])
        rmse = math.sqrt(mean_squared_error(y_true[:, i], y_pred[:, i]))
        r2   = r2_score(y_true[:, i], y_pred[:, i])
        mae_list.append(mae); rmse_list.append(rmse); r2_list.append(r2)

    mae_avg  = float(np.mean(mae_list))
    rmse_avg = float(np.mean(rmse_list))
    r2_avg   = float(np.mean(r2_list))
    S = 1.0 - mae_avg
    return {"loss": total_loss / max(1, n_batches), "mae_avg": mae_avg, "rmse_avg": rmse_avg, "r2_avg": r2_avg, "S": S}

def save_checkpoint(path, model, optimizer, epoch, best_S):
    torch.save({
        "epoch": epoch,
        "best_S": best_S,
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "label_cols": label_cols,
        "model_name": MODEL_NAME,
    }, path)

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

best_S = -1e18
pat = 0
history = []

print("Start training...")

for epoch in range(1, MAX_EPOCHS + 1):
    t0 = time.time()
    model.train()

    optimizer.zero_grad(set_to_none=True)
    running_loss, n_steps = 0.0, 0

    for step, batch in enumerate(train_loader, start=1):
        batch = to_device(batch, DEVICE)
        labels = batch["labels"]

        with torch.cuda.amp.autocast(enabled=use_amp):
            out = model(
                input_values=batch["input_values"],
                attention_mask=batch.get("attention_mask", None),
                labels=labels
            )
            loss = out["loss"] / GRAD_ACCUM_STEPS

        scaler.scale(loss).backward()
        running_loss += float(loss.item()) * GRAD_ACCUM_STEPS
        n_steps += 1

        if step % GRAD_ACCUM_STEPS == 0:
            if GRAD_CLIP is not None:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

    # flush akhir epoch kalau step tidak kelipatan
    if (len(train_loader) % GRAD_ACCUM_STEPS) != 0:
        if GRAD_CLIP is not None:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)

    train_loss = running_loss / max(1, n_steps)
    val = run_eval(model, val_loader)
    dt = time.time() - t0

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val["loss"],
        "val_mae_avg": val["mae_avg"],
        "val_rmse_avg": val["rmse_avg"],
        "val_r2_avg": val["r2_avg"],
        "val_S": val["S"],
        "sec": dt
    })

    print(
        f"[{epoch:02d}/{MAX_EPOCHS}] train_loss={train_loss:.5f} | "
        f"val_loss={val['loss']:.5f} | MAE={val['mae_avg']:.4f} | "
        f"RMSE={val['rmse_avg']:.4f} | R2={val['r2_avg']:.4f} | S={val['S']:.4f} | {dt:.1f}s"
    )

    if val["S"] > best_S + 1e-8:
        best_S = val["S"]
        pat = 0
        save_checkpoint(CKPT_DIR / "checkpoint_best.pt", model, optimizer, epoch, best_S)
        print(f"  -> saved best (S={best_S:.4f})")
    else:
        pat += 1
        print(f"  -> no improve | patience {pat}/{PATIENCE}")

    save_checkpoint(CKPT_DIR / "checkpoint_last.pt", model, optimizer, epoch, best_S)

    if pat >= PATIENCE:
        print("Early stopping triggered.")
        break

OUT_ROOT.mkdir(parents=True, exist_ok=True)
log_path = OUT_ROOT / "train_log.csv"
pd.DataFrame(history).to_csv(log_path, index=False)
print("Saved log:", log_path)
print("Best S:", best_S)
